# ICD Excel zu JSON Konvertierer

## Zweck

Dieses Notebook konvertiert den ICD-10-GM Katalog vom BARMER Institut für
Gesundheitsforschung (bifg) aus dem Excel-Format in strukturierte JSON-Dateien.

Die JSON-Dateien dienen als vorverarbeitete Datenbasis für die übrigen Notebooks
dieses Projekts. Gegenüber dem direkten Einlesen der Excel-Datei zur Laufzeit
sind sie deutlich performanter und bilden die dreistufige ICD-10-Hierarchie
(Kapitel → Gruppe → Code) bereits fertig strukturiert ab.

## Eingabe

- `ICD-10-GM-Kataloge_202005-2021.xlsx` mit je einem Tabellenblatt pro Jahr
- Relevante Spalten: B–S, ab Zeile 5 (erste vier Zeilen sind Metadaten)

## Ausgabe

- `icd_systematik_{year}.json` im zentralen Datenverzeichnis `DATA_DIR`

## Hinweis zur Nutzung

Dieses Notebook muss nur einmalig bzw. bei Aktualisierung des Katalogs
ausgeführt werden. Die erzeugten JSON-Dateien werden von den anderen
Notebooks des Projekts geladen.

## Anmerkung:
Die Datei, die durch dieses Notebook erstellt wird, wurde in dem Repository bereitgestellt, sodass das Notebook nicht ausgeführt werden muss.

In [1]:
from drg_risk import DATA_DIR
from drg_risk.data.dataclasses import ICDCode, ICDGroup, ICDChapter

import pandas as pd
import json
import os

In [ ]:
def convert_excel_to_json(excel_data: pd.DataFrame, year: int) -> None:
    """
    Überführt einen ICD-10-GM Jahreskatalog in eine JSON-Datei.

    Iteriert über alle Zeilen des DataFrames und baut die dreistufige
    ICD-10-Hierarchie (Kapitel → Gruppe → Codes) als verschachteltes
    Dictionary auf. Nicht kodierbare Endpunkte werden übersprungen.
    Das Ergebnis wird als JSON-Datei in DATA_DIR gespeichert.

    Parameters
    ----------
    excel_data : pd.DataFrame
        ICD-10-GM Katalog für ein einzelnes Jahr.
    year : int
        Kalenderjahr des Katalogs; wird im Dateinamen verwendet.
    """
    icd_systematik = {}

    for _, icd_code_element in excel_data.iterrows():
        # Nur kodierbare Endpunkte verwenden.
        if icd_code_element["Endpunkt"] == "kein kodierbarer Endpunkt":
            continue
        icd_code = ICDCode(
                code=icd_code_element.Code,
                description=icd_code_element["Code Bezeichnung"]
            )
        group_name = icd_code_element["Gruppe Bezeichnung"]
        start = icd_code_element["Gruppe von"]
        end = icd_code_element["Gruppe bis"]
        
        # Einstellige Gruppen (start == end) werden ohne Bereichsnotation gespeichert.
        if start == end:
            group_number = start
        else:
            group_number = f"{start}-{end}"

        chapter_name = icd_code_element["Kapitel Bezeichnung"]
        chapter_number = icd_code_element["Kapitel-Nr"]

        # Erstes Hinzufügen
        if chapter_name not in icd_systematik:
            icd_systematik[chapter_name] = ICDChapter(name=chapter_name, groups={}, number=chapter_number)

        # Erstes Hinzufügen
        if group_name not in icd_systematik[chapter_name].groups:
            icd_systematik[chapter_name].groups[group_name] = ICDGroup(name=group_name, codes=[], number=group_number)

        # kodierbarer Endpunkt hinzufügen
        icd_systematik[chapter_name].groups[group_name].codes.append(icd_code)

    # ICD-Systematik als JSON Datei speichern
    with open(os.path.join(DATA_DIR, f"icd_systematik_{year}.json"), "w", encoding="utf-8") as f:
        # ensure_ascii=False: Umlaute im JSON lesbar halten
        json.dump(icd_systematik, f, default=lambda o: o.__dict__, indent=4, ensure_ascii=False)

## Ausführung: Jahr 2010

Tabellenblatt `"2010"` aus der Excel-Datei einlesen und konvertieren.
`skiprows=4` überspringt die Metadatenzeilen

In [3]:
path = os.path.join(DATA_DIR, "ICD-10-GM-Kataloge_202005-2021.xlsx")

excel_data = pd.read_excel(path, sheet_name="2010", skiprows=4, usecols="B:S")

convert_excel_to_json(excel_data, 2010)